# FAS SINR Prediction

- **Author:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Imports

In [ ]:
# --------------------------- Standard Libraries ---------------------------- #
import os
import gc # Garbage Collector for Memory Management

# -------------------------------- Annotations ------------------------------- #
from typing import List, Optional, Tuple

# ------------------------- Data Processing Libraries ----------------------- #
import numpy as np
import matplotlib.pyplot as plt
import scipy.io

# ----------------------- TensorFlow and Keras Modules ---------------------- #
import tensorflow as tf
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard
from tensorflow.keras.optimizers import AdamW, SGD
from tensorflow.keras.backend import clear_session # Clean tf variables
from keras import layers, Model

# ---------------------------- Scikit-learn Modules ------------------------- #
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

### 1.2. GPU Configuration

In [ ]:
def get_gpu_info():
    """
    Retrieves and prints detailed GPU information including TensorFlow,
    CUDA, cuDNN versions, number of GPUs, and memory details.
    """
    # Display TensorFlow version
    print(f"TensorFlow Version: {tf.__version__}")

    # Check if TensorFlow is built with CUDA support and retrieve build info
    if tf.test.is_built_with_cuda():
        build_info = tf.sysconfig.get_build_info()
        print(f"TensorFlow is built with CUDA support")
        print(f"CUDA Version: {build_info['cuda_version']}")
        print(f"cuDNN Version: {build_info['cudnn_version']}")
    else:
        print("Running on CPU (No CUDA support detected)")

    # Detect available GPUs
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"\nNumber of GPUs detected: {len(gpus)}")
        print(f"Available GPU(s): {[gpu.name for gpu in gpus]}\n")
        tf.test.gpu_device_name()
    else:
        print("No GPUs found")
        print("Running on CPU")

# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

### 1.3. Folder and File Management

In [ ]:
def create_run_directory(base_dir: str = "runs", prefix: str = "training") -> str:
    """
    Creates a new directory for storing training logs, checkpoints, and plots.
    The directory name is based on the next available number.

    Args:
        base_dir (str): Base directory for storing training runs. Defaults to "runs".
        prefix (str): Prefix for the run directory. Defaults to "training".

    Returns:
        str: Path to the created run directory.
    """
    os.makedirs(base_dir, exist_ok=True)  # Ensure the base directory exists

    # Find the next available run number
    existing_dirs = [d for d in os.listdir(base_dir) if d.startswith(prefix) and d[len(prefix):].isdigit()]
    next_run_number = max([int(d[len(prefix):]) for d in existing_dirs] + [0]) + 1
    run_dir = os.path.join(base_dir, f"{prefix}{next_run_number}")
    os.makedirs(run_dir, exist_ok=True)  # Create the run directory

    return run_dir


# Create directories for the current run
RUN_DIR = create_run_directory(prefix="training_dnn_")
CHECKPOINT_DIR = os.path.join(RUN_DIR, "weights")
TENSORBOARD_DIR = os.path.join(RUN_DIR, "tensorboard")
PLOT_DIR = os.path.join(RUN_DIR, "figures")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TENSORBOARD_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print(f"Run directory: {RUN_DIR}")
print(f"Checkpoints directory: {CHECKPOINT_DIR}")
print(f"TensorBoard Logs directory: {TENSORBOARD_DIR}")
print(f"Plots directory: {PLOT_DIR}")

## 2. Constants and Hyperparameters

In [ ]:
# ---------------------------- Dataset Parameters ---------------------------- #
TOTAL_NUM_PORTS = 144  # Total number of ports in the dataset

# ---------------------------- Training Parameters --------------------------- #
BATCH_SIZE = 32  # Batch size for training
EPOCHS = 20  # Number of epochs for training

observed_ports_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# ------------------------- Randomization Parameters ------------------------- #
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 3. Utility Functions

### 3.1. Plotting Functions

In [ ]:
def plot_scientific(
    x: List[float],
    y_datasets: List[List[float]],
    labels: Optional[List[str]] = None,
    x_label: str = "X-axis",
    y_label: str = "Y-axis",
    title: str = "Scientific Plot",
    x_integer: bool = False,
    y_log: bool = False,
    markers: Optional[List[str]] = None,
    line_styles: Optional[List[str]] = None,
    legend_loc: str = "best",
    grid: bool = True,
    xlim: Optional[Tuple[float, float]] = None,
    ylim: Optional[Tuple[float, float]] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
) -> None:
    """
    Plots multiple Y datasets against a shared X-axis with scientific paper styling.

    Args:
        x (List[float]): List of X-axis values.
        y_datasets (List[List[float]]): List of lists containing Y-axis datasets.
        labels (Optional[List[str]]): Labels for each dataset for the legend.
        x_label (str): Label for the X-axis. Default is "X-axis".
        y_label (str): Label for the Y-axis. Default is "Y-axis".
        title (str): Title of the plot. Default is "Scientific Plot".
        x_integer (bool): Force X-axis to display only integer values. Default is False.
        y_log (bool): Use logarithmic scale for the Y-axis. Default is False.
        markers (Optional[List[str]]): List of marker styles for each dataset.
        line_styles (Optional[List[str]]): List of line styles for each dataset.
        legend_loc (str): Location of the legend. Default is "best".
        grid (bool): Whether to display a grid. Default is True.
        xlim (Optional[Tuple[float, float]]): Limits for the X-axis as (min, max). Default is None.
        ylim (Optional[Tuple[float, float]]): Limits for the Y-axis as (min, max). Default is None.
        save_path (Optional[str]): Path to save the plot as a file. Default is None.
        dpi (int): Resolution of the saved plot in dots per inch. Default is 300.

    Returns:
        None: Displays the plot and optionally saves it as an image.
    """
    # Validate input dimensions
    if any(len(y) != len(x) for y in y_datasets):
        raise ValueError("All Y datasets must have the same length as the X dataset.")

    # Initialize the plot
    plt.figure(figsize=(8, 6))

    # Plot each dataset
    for i, y in enumerate(y_datasets):
        label = labels[i] if labels and i < len(labels) else f"Dataset {i + 1}"
        marker = markers[i] if markers and i < len(markers) else "o"
        line_style = line_styles[i] if line_styles and i < len(line_styles) else "-"
        plt.plot(x, y, label=label, marker=marker, linestyle=line_style)

    # Configure axes and title
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label, fontsize=12)
    plt.title(title, fontsize=14, weight="bold")
    plt.legend(loc=legend_loc, fontsize=10)

    # Configure axis limits
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)

    # Configure X-axis for integers only
    if x_integer:
        plt.xticks(ticks=range(int(min(x)), int(max(x)) + 1))

    # Enable logarithmic scale for Y-axis if requested
    if y_log:
        plt.yscale("log")

    # Add grid if requested
    if grid:
        plt.grid(visible=True, linestyle="--", linewidth=0.5, alpha=0.7)

    # Final styling
    plt.tight_layout()

    # Save plot if path is provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, format="png")

    # Show plot
    plt.show()
    

def plot_histogram(
    datasets: List[List[float]],
    bins: int = 10,
    density: bool = False,
    labels: Optional[List[str]] = None,
    colors: Optional[List[str]] = None,
    edgecolors: Optional[List[str]] = None,
    alpha: float = 0.7,
    x_label: str = "X-axis",
    y_label: str = "Frequency",
    title: str = "Histogram",
    legend_loc: str = "best",
    grid: bool = True,
    xlim: Optional[Tuple[float, float]] = None,
    ylim: Optional[Tuple[float, float]] = None,
    save_path: Optional[str] = None,
    dpi: int = 300,
) -> None:
    """
    Plots a histogram for one or more datasets with scientific styling.

    Args:
        datasets (List[List[float]]): List of datasets to plot histograms for.
        bins (int): Number of bins in the histogram. Default is 10.
        density (bool): If True, normalizes the histogram so the area equals 1. Default is False.
        labels (Optional[List[str]]): Labels for each dataset for the legend. Default is None.
        colors (Optional[List[str]]): Colors for each dataset. Default is None.
        edgecolors (Optional[List[str]]): Edge colors for each dataset. Default is None.
        alpha (float): Transparency level of bars (0: fully transparent, 1: opaque). Default is 0.7.
        x_label (str): Label for the X-axis. Default is "X-axis".
        y_label (str): Label for the Y-axis. Default is "Frequency".
        title (str): Title of the histogram. Default is "Histogram".
        legend_loc (str): Location of the legend. Default is "best".
        grid (bool): Whether to display a grid. Default is True.
        xlim (Optional[Tuple[float, float]]): Limits for the X-axis as (min, max). Default is None.
        ylim (Optional[Tuple[float, float]]): Limits for the Y-axis as (min, max). Default is None.
        save_path (Optional[str]): Path to save the histogram as a file. Default is None.
        dpi (int): Resolution of the saved histogram in dots per inch. Default is 300.

    Returns:
        None: Displays the histogram and optionally saves it as an image.
    """
    # Initialize the plot
    plt.figure(figsize=(8, 6))

    # Plot each dataset as a histogram
    for i, data in enumerate(datasets):
        label = labels[i] if labels and i < len(labels) else f"Dataset {i + 1}"
        color = colors[i] if colors and i < len(colors) else None
        edgecolor = edgecolors[i] if edgecolors and i < len(edgecolors) else None
        plt.hist(
            data,
            bins=bins,
            density=density,
            alpha=alpha,
            label=label,
            color=color,
            edgecolor=edgecolor,
        )

    # Configure axes and title
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label if not density else "Density", fontsize=12)
    plt.title(title, fontsize=14, weight="bold")

    # Configure axis limits
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)

    # Add grid if requested
    if grid:
        plt.grid(visible=True, linestyle="--", linewidth=0.5, alpha=0.7)

    # Add legend if labels are provided
    if labels:
        plt.legend(loc=legend_loc, fontsize=10)

    # Final styling
    plt.tight_layout()

    # Save histogram if path is provided
    if save_path:
        plt.savefig(save_path, dpi=dpi, format="png")

    # Show histogram
    plt.show()

## 4. Data Loading and Preprocessing

In [ ]:
# --------------------- Load the dataset in matlab format -------------------- #
dataset = (scipy.io.loadmat('data/Rayleigh_3dB/SNR_events.mat')['SNR_events']).T

print(f"Original dataset shape: {dataset.shape}")

# ---------------------------- Data Configuration ---------------------------- #
# Subsample data
dataset = dataset[:int(0.5 * dataset.shape[0]), :] # 1.0 means full dataset

# Changing the scale of the data
# dataset = np.log10(dataset)
#! Warning: this makes the numbers much closer, which may make training harder

print(f"Shape of the data after configuration: {dataset.shape}\n")

# ------------------------------- Checking data ------------------------------ #
print(f"NaNs in data: {np.isnan(dataset).any()}")
print(f"Infs in data: {np.isinf(dataset).any()}")
print(f"Max value: {np.max(dataset)}, Min value: {np.min(dataset)}\n")

# ---------------------------- Compute statistics ---------------------------- #
mean_of_rows = np.mean(dataset, axis=0)
mean_of_means = np.mean(mean_of_rows)
print(f"Mean of values in the dataset: {mean_of_means}")

In [ ]:
# ---------------------------- Plot the histogram ---------------------------- #
plot_histogram(
    datasets=[dataset],
    bins=100,
    edgecolors=["black"],
    title="Histogram of Data",
    x_label="SNR",
    y_label="Frequency",
    save_path=os.path.join(PLOT_DIR, "data_histogram.png"),
)

## 5. Model Definition

In [ ]:
def build_dnn(X: np.ndarray, X_test: np.ndarray, input_shape: int, output_units: int) -> Tuple[tf.keras.Model, np.ndarray]:
    """
    Builds a DNN model and preprocesses the data based on the input shape.

    Args:
        X (np.ndarray): Input features (combined training and testing).
        X_test (np.ndarray): Testing features.
        input_shape (int): Input size (number of observed ports).
        output_units (int): Output size (number of total ports).

    Returns:
        Tuple[tf.keras.Model, np.ndarray]: The built model and the preprocessed input features.
    """
    # -------------------- Choose scaler based on input shape -------------------- #
    scalers = {
        1: MinMaxScaler(feature_range=(-1, 1)),
        2: StandardScaler(),
        3: MinMaxScaler(feature_range=(0, 1)),
        4: StandardScaler(),
        5: StandardScaler(),
        6: MinMaxScaler(feature_range=(0, 1)),
        7: MinMaxScaler(feature_range=(-1, 1)),
        8: MinMaxScaler(feature_range=(-1, 1)),
        9: MinMaxScaler(feature_range=(-1, 1)),
        10: StandardScaler()
    }
    
    scaler = scalers.get(input_shape, StandardScaler())
    X = scaler.fit_transform(X)
    X_test = scaler.transform(X_test)

    # ----------------------------- Define the model ----------------------------- #
    inputs = layers.Input(shape=(input_shape,))
    
    if input_shape == 1:
        x = layers.Dense(units=512, activation="tanh")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.0)(x)
        x = layers.Dense(units=432, activation="relu")(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(units=320, activation="tanh")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Dense(units=224, activation="tanh")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.0)(x)
    elif input_shape == 2:
        x = layers.Dense(units=256, activation="relu")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)
        x = layers.Dense(units=224, activation="tanh")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
    elif input_shape == 3:
        x = layers.Dense(units=2048, activation="tanh")(inputs)
        x = layers.Dropout(0.1)(x)
        x = layers.Dense(units=1104, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)
    elif input_shape == 4:
        x = layers.Dense(units=1024, activation="relu")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.0)(x)
        x = layers.Dense(units=544, activation="tanh")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.0)(x)
    elif input_shape == 5:
        x = layers.Dense(units=1024, activation="relu")(inputs)
        x = layers.Dropout(0.4)(x)
        x = layers.Dense(units=784, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.1)(x)
    elif input_shape == 6:
        x = layers.Dense(units=2048, activation="relu")(inputs)
        x = layers.Dropout(0.0)(x)
        x = layers.Dense(units=2048, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.1)(x)
        x = layers.Dense(units=2016, activation="relu")(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(units=1968, activation="tanh")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
    elif input_shape == 7:
        x = layers.Dense(units=256, activation="tanh")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.0)(x)
        x = layers.Dense(units=176, activation="tanh")(x)
        x = layers.Dropout(0.0)(x)
    elif input_shape == 8:
        x = layers.Dense(units=512, activation="relu")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(units=272, activation="tanh")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.1)(x)
    elif input_shape == 9:
        x = layers.Dense(units=1280, activation="tanh")(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.1)(x)
        x = layers.Dense(units=1072, activation="tanh")(x)
        x = layers.Dropout(0.5)(x)
    elif input_shape == 10:
        x = layers.Dense(units=1792, activation="tanh")(inputs)
        x = layers.Dropout(0.0)(x)
        x = layers.Dense(units=1600, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)
    
    outputs = layers.Dense(output_units, activation="linear")(x)
    model = Model(inputs=inputs, outputs=outputs)

    return model, X, X_test


## 6. Other Functions

In [ ]:
def get_observed_ports(sinr_data, num_observed_ports, total_ports):
    """
    Extracts SINR values for the specified number of observed ports.

    The function selects a subset of SINR data by identifying equally spaced ports based on the
    number of observed ports specified. It returns the SINR values for these observed ports and
    their corresponding indices.

    Args:
        sinr_data (numpy.ndarray): A 2D array where each row represents an observation and each column
                                   represents a port with its corresponding SINR values.
        num_observed_ports (int): The number of observed ports to select from the SINR data.
        total_ports (int): The total number of ports in the SINR data.

    Returns:
        observed_sinr (numpy.ndarray): A 2D array containing the SINR values for the observed ports.
        observed_indices (numpy.ndarray): A 1D array of the indices corresponding to the observed ports.
    """
    observed_indices = np.linspace(0, total_ports - 1, num_observed_ports, dtype=int)
    observed_sinr = sinr_data[:, observed_indices]

    return observed_sinr, observed_indices

### 6.3. Tests

In [ ]:
# Test the function
observed_sinr, observed_indices = get_observed_ports(
    dataset, num_observed_ports=10, total_ports=TOTAL_NUM_PORTS
)
print(f"Observed SINR shape: {observed_sinr.shape}")
print(f"Observed port indices: {observed_indices}")

## 7. Training

In [ ]:
# Lists to store results
train_mse_history_list = []
val_mse_history_list = []

for n in observed_ports_list:
    print("\n---------------------------------------------")
    print(f"Training model with {n} observed ports...")

    # ----------------------------- Data Preparation ----------------------------- #
    observed_ports, _ = get_observed_ports(
        sinr_data=dataset, num_observed_ports=n, total_ports=TOTAL_NUM_PORTS
    )

    X_train, X_test, y_train, y_test = train_test_split(
        observed_ports,  # X
        dataset,  # y
        test_size=0.2,
        random_state=SEED,
    )

    # --------------------------------- Callbacks -------------------------------- #
    tensorboard_callback = TensorBoard(log_dir=TENSORBOARD_DIR, histogram_freq=1)
    # Run TensorBoard in the terminal using this command: tensorboard --logdir logs/fit
    # Run the command at the folder containing the logs directory

    checkpoint_callback = ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, f"best_model_observed_ports_{n}.weights.h5"),
        save_weights_only=True,  # Save only the weights
        monitor="val_loss",  # Use validation MSE to monitor the best model
        mode="min",
        save_best_only=True,  # Save only the best model
        verbose=0,  # Verbosity level for saving process
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",  # The metric to monitor
        mode="min",  # Maximize or minimize the metric
        patience=3,  # Number of epochs with no improvement before stopping
        verbose=0,  # Verbosity level (0 = silent, 1 = report stopping)
        restore_best_weights=True,  # Restore the best weights at the end of training
    )

    # ------------------------- Build and Train the model ------------------------ #
    optimizer = SGD(learning_rate=0.001)

    model, X_train, X_test = build_dnn(X=X_train, X_test=X_test, input_shape=n, output_units=TOTAL_NUM_PORTS)
    model.compile(optimizer=optimizer, loss="mse")

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        callbacks=[
            # early_stopping,
            checkpoint_callback,
            # tensorboard_callback,
        ],
        verbose=1,
    )
    
    train_mse_history_list.append(history.history["loss"])  # Store loss values
    val_mse_history_list.append(history.history["val_loss"])  # Store loss values

    # Clean up tf variables and free up memory
    clear_session()
    gc.collect()

## 8. Results

### 8.1. Plot MSE for each model

In [ ]:
for idx, n in enumerate(observed_ports_list):
    plot_scientific(
        x=range(1, EPOCHS + 1),
        y_datasets=[train_mse_history_list[idx], val_mse_history_list[idx]],
        labels=["Training MSE", "Validation MSE"],
        x_label="Epoch",
        y_label="Mean Squared Error (MSE)",
        title=f"Training and Validation MSE with {n} Observed Ports",
        markers=["o", "x"],
        line_styles=["-", "--"],
        ylim=(
            0,
            max(max(train_mse_history_list[idx]), max(val_mse_history_list[idx])) * 1.05,
        ),
        grid=True,
        x_integer=True,
        save_path=os.path.join(PLOT_DIR, f"mse_{n}_observed_ports.png"),
    )

### 8.2. Plot the Merged MSE over epochs

In [ ]:
labels = [f"{ports} Observed Ports" for ports in observed_ports_list]

plot_scientific(
    x=range(1, EPOCHS + 1),
    y_datasets=val_mse_history_list,
    labels=labels,
    x_label="Epoch",
    y_label="Mean Squared Error (MSE)",
    title="MSE Over Epochs for Different Observed Ports",
    markers=["o"] * len(val_mse_history_list),  # Default marker for all datasets
    line_styles=["-"] * len(val_mse_history_list),  # Default line style for all datasets
    legend_loc="upper right",  # Adjust legend position for clarity
    ylim=(
        0,
        max(max(mse) for mse in val_mse_history_list) * 1.05,
    ),
    grid=True,
    x_integer=True,
    save_path=os.path.join(PLOT_DIR, "merged_mse.png"),
)